# **Arquitetura de Software e Armazenamento**
<font size=3>
    
O objetivo desta fase é unificar todos os módulos desenvolvidos nas fases anteriores (leitura de Wi-Fi, trilateração, algoritmos de rotas e síntese de voz) em uma arquitetura de código profissional, escalável, reativa e capaz de funcionar **100% offline** sem dependência de internet.


### **1. Padrão de Arquitetura MVVM (*Model-View-ViewModel*):**
<font size=3>

Para evitar que o aplicativo trave ou fique desorganizado, adotamos a arquitetura **MVVM**, recomendada oficialmente pelo Google para desenvolvimento Android.

```
       +--------------------------------------------------+
       |                  VIEW (UI / VUI)                 |
       | Activity / Fragment / TextToSpeech / TalkBack    |
       +------------------------+-------------------------+
                                | Observa estados (Flow)
                                v
       +--------------------------------------------------+
       |                    VIEWMODEL                     |
       | Gerencia o Estado da Rota e Coordenadas Atuais   |
       +------------------------+-------------------------+
                                | Chama regras de negócio
                                v
       +--------------------------------------------------+
       |                  MODEL / REPOSITORY              |
       |  Room DB | Algoritmo A* | Motor IPS | Wi-Fi Scan |
       +--------------------------------------------------+

```

* **Model (Camada de Dados e Regras de Negócio):** Contém os bancos de dados local, os algoritmos de trilateração, o cálculo de rotas $A^*$ e os serviços de hardware (Wi-Fi e Sensores).

* **ViewModel (Gerenciador de Estado):** Mantém e expõe o estado atual da navegação (ex: posição $(X,Y)$ atual, próximo passo da rota, status do sinal Wi-Fi). Sobrevive a mudanças de configuração (como rotação de tela).

* **View (Interface e Saída de Áudio):** Escuta as alterações do ViewModel e dispara as falas do `TextToSpeech`, as vibrações ou atualiza a tela do usuário.


### **2. Banco de Dados Local com Room (*Offline-First*):**
<font size=3>

Como os prédios podem ter pontos de "sombra" sem sinal de internet móvel (4G/5G), todo o mapa do prédio (roteadores, nós, arestas e POIs) deve ficar armazenado localmente no celular via biblioteca **Room** (abstração do SQLite nativo).

#### **2.1 Mapeamento da Entidade de Roteadores no Room:**

In [ ]:
// ATENÇÃO: o pacote androidx.* pertence exclusivamente ao sistema operacional e SDK do Android, o Jupyter-Lab irá emitir uma mensagem de erro.
// Assim, faça as adaptações necessárias para rodar no notebook.
import androidx.room.Entity
import androidx.room.PrimaryKey

In [ ]:
@Entity(tableName = "wifi_routers")
data class WifiRouterEntity(@PrimaryKey val bssid: String,
                            val ssid: String,
                            val x: Double,
                            val y: Double,
                            val floor: Int,
                            val referenceRssi1m: Int,
                            val attenuationExponent: Double,
                            val buildingId: String)


#### **2.2 Interface DAO (*Data Access Object*) para Consulta Local:**

In [ ]:
import androidx.room.Dao
import androidx.room.Insert
import androidx.room.OnConflictStrategy
import androidx.room.Query
import kotlinx.coroutines.flow.Flow

In [ ]:
@Dao
interface MapDao {
    @Insert(onConflict = OnConflictStrategy.REPLACE)
    suspend fun saveWifiRouters(routers: List<WifiRouterEntity>)

    @Query("SELECT * FROM wifi_routers WHERE buildingId = :buildingId")
    fun getWifiRoutersByBuilding(buildingId: String): Flow<List<WifiRouterEntity>>
}

### **3. Programação Reativa com Kotlin Coroutines e Flow:**
<font size=3>

O aplicativo precisa executar múltiplas tarefas assíncronas concorrentes sem congelar a interface de usuário nem interromper a narração de voz:

* **Coroutines (`Dispatchers.IO`):** Executam o processamento pesado (trilateração matemática, consultas ao banco Room e algoritmo $A^*$) em threads secundárias de segundo plano.

* **`StateFlow` e `SharedFlow`:** Estruturas de dados reativas que transmitem atualizações constantes de localização e instruções em tempo real da ViewModel para a View.

In [ ]:
import androidx.lifecycle.ViewModel
import androidx.lifecycle.viewModelScope
import kotlinx.coroutines.Dispatchers
import kotlinx.coroutines.flow.MutableStateFlow
import kotlinx.coroutines.flow.StateFlow
import kotlinx.coroutines.flow.asStateFlow
import kotlinx.coroutines.launch

In [ ]:
// Estado da UI/VUI exposto para a View
data class NavigationState(val positionX: Double = 0.0,
                           val positionY: Double = 0.0,
                           val currentVoiceInstruction: String = "Aguardando sinal Wi-Fi...",
                           val isNavigating: Boolean = false)

class NavigationViewModel(private val mapDao: MapDao) : ViewModel() {

    private val _state = MutableStateFlow(NavigationState())
    val state: StateFlow<NavigationState> = _state.asStateFlow()

    fun processNewWifiReadings(rssiReadings: Map<String, Int>) {
        // Executa o cálculo de trilateração em background thread
        viewModelScope.launch(Dispatchers.IO) {
            // 1. Simulação do cálculo matemático do Motor IPS
            val newPositionX = 12.5
            val newPositionY = 8.0

            // 2. Atualização reativa do estado
            _state.value = _state.value.copy(
                positionX = newPositionX,
                positionY = newPositionY,
                currentVoiceInstruction = "Siga em frente por 5 metros."
            )
        }
    }
}

### **4. Repositório Unificado de Navegação (*Repository Pattern*):**
<font size=3>

O **Repository** centraliza o fluxo de dados do aplicativo. A ViewModel não precisa saber se os dados do mapa vieram do arquivo JSON baixado previamente ou do banco local Room; ela apenas solicita as informações ao `NavegacaoRepository`.

In [ ]:
class NavigationRepository(private val mapDao: MapDao) {

    suspend fun loadAndSaveInitialMap(map: BuildingMap) {
        val routerEntities = map.accessPoints.map { ap ->
            WifiRouterEntity(bssid = ap.bssid,
                             ssid = ap.ssid,
                             x = ap.x,
                             y = ap.y,
                             floor = ap.floor,
                             referenceRssi1m = ap.referenceRssi1m,
                             attenuationExponent = ap.attenuationExponent,
                             buildingId = map.buildingId)
        }
        mapDao.saveWifiRouters(routerEntities)
    }
}

### **Atividade prática:**
<font size=3>
    
Estruturar a Arquitetura com Reatividade (StateFlow):\
Crie um fluxo reativo completo onde a View reage automaticamente às mudanças do motor de navegação sem acoplamento direto.

1. **Passo 1:** Configure as dependências do Room e Coroutines no `build.gradle.kts`;
    
    <details>
    <summary>Resposta:</summary>
    
    <font size=2>
    <br>
        
    ```kotlin
    implementation("androidx.room:room-runtime:2.6.1")
    implementation("androidx.room:room-ktx:2.6.1")
    kapt("androidx.room:room-compiler:2.6.1")
    implementation("androidx.lifecycle:lifecycle-viewmodel-ktx:2.7.0")
    
    ```
    
    <br>

    </details>
    <br>
   
3. **Passo 2:** Crie a `MainActivity.kt` e conecte-a com o `NavegacaoViewModel` utilizando `lifecycleScope` para observar as alterações no `StateFlow`.

    <details>
    <summary>Resposta:</summary>
    
    <font size=2>
    <br>
            
    ```kotlin
    class MainActivity : AppCompatActivity() {
    
        private val viewModel: NavigationViewModel by viewModels()
        private lateinit var voiceManager: VoiceManager
    
        override fun onCreate(savedInstanceState: Bundle?) {
            super.onCreate(savedInstanceState)
            setContentView(R.layout.activity_main)
    
            voiceManager = VoiceManager(this)
    
            // Observa o fluxo de estado de forma reativa
            lifecycleScope.launch {
                repeatOnLifecycle(Lifecycle.State.STARTED) {
                    viewModel.state.collect { currentState ->
                        // Toda vez que a posição ou instrução mudar, atualiza a voz e a tela
                        voiceManager.speak(currentState.currentVoiceInstruction)
                    }
                }
            }
        }
    }
    
    ```
    
    <br>

    </details>
